# 🏗️ Notebook 1: Key-Value Store — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A **distributed key-value store** like Amazon DynamoDB, Apache Cassandra, or Riak.
It exposes three tiny operations — `put(k, v)`, `get(k)`, `delete(k)` — but runs across
many machines, survives node failures, and scales horizontally.

Think of it as a giant Python dictionary that:
- lives on **many computers** (not one),
- **doesn't lose data** when a machine dies,
- keeps answering requests when some nodes are **slow or offline**.

### Functional requirements
- `put(key, value)`, `get(key)`, `delete(key)`
- **Tunable consistency**: caller chooses speed vs correctness per request
- Data must **survive single-node failure**

### Non-functional
- **Horizontally scalable** — add nodes, capacity grows linearly.
- **Highly available** — reads/writes succeed even when some nodes are down.
- **Low latency** — under ~10 ms p99 for small values.
- **Durable** — once a write is ack'd, don't lose it.

### The three key design questions
1. **Which node stores key `k`?** → *Partitioning* (consistent hashing)
2. **How do we survive failures?** → *Replication* (N copies)
3. **What if replicas disagree?** → *Consistency* (W/R quorums, conflict resolution)


## 🏛️ High-level architecture

```
            ┌──────────┐
  client →  │ any node │  ← acts as "coordinator" for this request
            └────┬─────┘
                 │  1. hash(key) → find N replicas on the ring
                 │  2. forward put/get to them
                 ▼
       ┌─────────┼─────────┐
       ▼         ▼         ▼
    ┌─────┐   ┌─────┐   ┌─────┐
    │ N1  │   │ N2  │   │ N3  │   ← replicas (actual storage)
    └─────┘   └─────┘   └─────┘
         ↖  gossip / anti-entropy ↗
```

**Key insight:** every node is equal. Any node can be the coordinator. There is
no single master. That's how we stay available when machines die.


## 😱 The naive approach (bad practice)

Let's start with the obvious but broken idea: pick the node for a key with
`node_index = hash(key) % N`. It works… until `N` changes.


In [1]:
import hashlib

def h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)

def node_mod_n(key: str, nodes: list[str]) -> str:
    return nodes[h(key) % len(nodes)]

NODES_4 = ["n1", "n2", "n3", "n4"]
NODES_5 = ["n1", "n2", "n3", "n4", "n5"]   # added n5

keys = [f"user:{i}" for i in range(10_000)]
moved = sum(1 for k in keys if node_mod_n(k, NODES_4) != node_mod_n(k, NODES_5))
print(f"hash % N  → adding ONE node moves {moved}/10000 keys ({moved/100:.1f}%)")
print("That's a full data reshuffle across the cluster. Unusable in practice.")

hash % N  → adding ONE node moves 7960/10000 keys (79.6%)
That's a full data reshuffle across the cluster. Unusable in practice.


Adding one node remaps ~80% of keys. In a real cluster every one of those keys
would have to be copied between machines — hours of network traffic, cache misses,
and hot spots. This is why nobody uses plain `hash % N`.


## ✅ The best practice: consistent hashing

Place nodes and keys on a ring of positions `0 … 2^32-1`. A key belongs to the
**first node clockwise** from its position. Adding or removing a node only
disturbs the slice between it and its neighbor — typically `1/N` of the data.

```
     0 ───────── 2^32 ─── (the ring)

  place each node at hash(node_id) on the ring
  place each key  at hash(key)     on the ring
  key is owned by the FIRST node clockwise from its position.

        (N1)
         ●
        /
       /          ● (N4)
      /
 ────●            ●────   ring
      \
       \
        ● (N2)   ● (N3)

  adding N5 only steals a slice of one neighbor's range.
```

### Virtual nodes (vnodes)
A single physical node claims **many** positions on the ring (e.g., 64–256 "vnodes").
This smooths out load imbalance — otherwise one unlucky node could own a huge arc
of the ring.

We'll implement this in **Notebook 3**. Sneak preview:


In [2]:
# Sneak preview: consistent hashing moves only ~1/N keys on cluster change
# (full implementation + math in notebook 3)
import bisect, hashlib
def H(s): return int(hashlib.md5(s.encode()).hexdigest(), 16)

def build_ring(nodes, vnodes=64):
    ring = sorted((H(f"{n}#{i}"), n) for n in nodes for i in range(vnodes))
    return ring, [p for p, _ in ring]

def owner(ring, positions, key):
    i = bisect.bisect_right(positions, H(key)) % len(ring)
    return ring[i][1]

r4 = build_ring(["n1","n2","n3","n4"])
r5 = build_ring(["n1","n2","n3","n4","n5"])
keys = [f"user:{i}" for i in range(10_000)]
moved = sum(1 for k in keys if owner(*r4, k) != owner(*r5, k))
print(f"consistent hashing → adding ONE node moves only {moved}/10000 keys ({moved/100:.1f}%)")
print(f"Theoretical fair share is 1/5 = 20%. We're close. 🎯")

consistent hashing → adding ONE node moves only 2068/10000 keys (20.7%)
Theoretical fair share is 1/5 = 20%. We're close. 🎯


## Replication & the W/R/N knobs

For each key we store **N copies** on N consecutive nodes on the ring (e.g., N=3).
Two knobs control consistency:

- **W** = min replicas that must ack a write before we tell the client "OK"
- **R** = min replicas that must answer a read before we return to the client

**Strong-consistency guarantee:** `R + W > N` ⇒ read set must overlap with write set,
so any successful read sees at least one copy of the latest write.

| N | W | R | Flavor | Who uses it |
|---|---|---|--------|-------------|
| 3 | 1 | 1 | Fast, eventually consistent | DynamoDB default, Cassandra `ONE` |
| 3 | 2 | 2 | Quorum — survives 1 replica outage | Cassandra `QUORUM` (recommended) |
| 3 | 3 | 1 | Fast reads, slow writes | read-heavy workloads |
| 3 | 1 | 3 | Fast writes, slow reads | write-heavy, rare reads |

### CAP in plain English
During a network **P**artition you must pick one:
- **C**onsistency: refuse requests you can't answer correctly (smaller W+R won't satisfy quorum).
- **A**vailability: answer anyway, possibly with stale data.

DynamoDB-style stores (and this lab) lean **AP** with tunable consistency.


## 🌍 Real-world systems at a glance

| System | Partitioning | Replication | Consistency default | Notes |
|---|---|---|---|---|
| **DynamoDB** (AWS) | Consistent hashing | N=3 across AZs | Eventual, optional strong | Closed source, inspired this whole design |
| **Cassandra** | Consistent hashing + vnodes | Tunable N, W, R | Quorum (usually) | Open source; gossip; hinted handoff |
| **Riak** | Consistent hashing + vnodes | Tunable | AP, vector clocks | Very faithful Dynamo clone |
| **Redis Cluster** | Hash slots (16384 fixed) | Primary + replicas | Strong on primary | Not consistent hashing — fixed slot map |
| **etcd / Consul** | Raft log (single group) | N=3 or 5 | Strong (CP) | Configuration store, not a KV cache |
| **Memcached** | Client-side hashing | **None** | N/A | Cache only, data loss tolerated |

👉 **Takeaway:** the design we're building in this lab matches DynamoDB / Cassandra / Riak.
It is **not** how etcd or Redis Cluster work — those make different tradeoffs
(strong consistency via Raft, or fixed slot maps).


## Recap

- `hash % N` is a trap: adding a node reshuffles ~`(N-1)/N` of the data.
- Consistent hashing + vnodes limits movement to `~1/N`.
- Replicate each key to N nodes; tune W and R for consistency vs availability.
- `R + W > N` ⇒ strong consistency; otherwise eventual.
- Real systems (Dynamo, Cassandra, Riak) all use this same skeleton.

Next notebook: data model, versioning, and API design.
